# Pràctica 4: Embeddings Distribucionals i Contextuals per a Similitud en Espanyol

**Objectiu**: Entrenar i avaluar models d'embeddings (Word2Vec, fastText, BERT) per a tasques de similitud semàntica en espanyol.

**Datasets**:
- **Multi-SimLex (SPA)**: parells de paraules amb puntuacions de similitud → avaluació intrínseca (Spearman)
- **Spanish STS**: parells de frases amb puntuacions de similitud → avaluació extrínseca (Pearson)

## 0. Instal·lació de dependències

In [ ]:
!pip install gensim datasets transformers torch scikit-learn scipy numpy pandas tqdm

## 1. Descàrrega i Preprocessament del Corpus

In [ ]:
import os
import re
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

In [ ]:
# Descàrrega del corpus wiki en espanyol
# NOTA: és un fitxer gran (~1.5GB comprimit). Pot trigar uns minuts.
!wget -nc https://www.cs.upc.edu/~nlp/wikicorpus/raw.es.tgz
!tar -xzf raw.es.tgz

In [ ]:
# Llistar els fitxers extrets
import glob
corpus_files = sorted(glob.glob('raw.es*'))
print(f"Fitxers trobats: {corpus_files[:5]} ...")

In [ ]:
def preprocess_line(line):
    """Tokenitza i neteja una línia de text en espanyol."""
    # Convertir a minúscules
    line = line.lower()
    # Eliminar caràcters no alfabètics (excepte accents i ñ)
    line = re.sub(r'[^a-záéíóúàèìòùüñ\s]', ' ', line)
    # Eliminar espais múltiples
    line = re.sub(r'\s+', ' ', line).strip()
    return line.split()


class CorpusIterator:
    """
    Iterador sobre el corpus wiki. Permet iterar múltiples vegades
    sense carregar tot en memòria.
    
    Args:
        files: llista de fitxers del corpus
        max_sentences: límit de frases (None = tot el corpus)
    """
    def __init__(self, files, max_sentences=None):
        self.files = files
        self.max_sentences = max_sentences
    
    def __iter__(self):
        count = 0
        for filepath in self.files:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    tokens = preprocess_line(line)
                    if len(tokens) >= 3:  # Ignorar frases molt curtes
                        yield tokens
                        count += 1
                        if self.max_sentences and count >= self.max_sentences:
                            return


# Versió petita per a proves ràpides (100K frases)
# Canviar max_sentences=None per usar tot el corpus
corpus_small = CorpusIterator(corpus_files, max_sentences=100_000)
corpus_full  = CorpusIterator(corpus_files, max_sentences=None)

print("Iteradors de corpus creats.")
print("Exemple de primeres frases:")
for i, sent in enumerate(corpus_small):
    print(sent[:10])
    if i >= 2:
        break

## 2. Entrenament d'Embeddings Estàtics

Compararem:
- **Word2Vec** (Skip-Gram) amb dimensions 25, 50, 100
- **fastText** amb dimensions 100
- **fastText oficial** (pre-entrenat sobre CommonCrawl)

In [ ]:
from gensim.models import Word2Vec, FastText

# ─────────────────────────────────────────────
# Word2Vec – diverses dimensions
# ─────────────────────────────────────────────
w2v_configs = [
    {'vector_size': 25,  'name': 'w2v_25'},
    {'vector_size': 50,  'name': 'w2v_50'},
    {'vector_size': 100, 'name': 'w2v_100'},
]

w2v_models = {}

for cfg in w2v_configs:
    print(f"\nEntrenant Word2Vec dim={cfg['vector_size']}...")
    model = Word2Vec(
        sentences=CorpusIterator(corpus_files, max_sentences=100_000),
        vector_size=cfg['vector_size'],
        window=5,
        min_count=5,
        workers=4,
        sg=1,          # Skip-Gram
        epochs=5,
        seed=42
    )
    model.save(f"{cfg['name']}.model")
    w2v_models[cfg['name']] = model
    vocab_size = len(model.wv)
    print(f"  Vocabulari: {vocab_size:,} paraules")

print("\nEntrenament Word2Vec completat!")

In [ ]:
# ─────────────────────────────────────────────
# fastText propi (100 dimensions)
# ─────────────────────────────────────────────
print("Entrenant fastText dim=100...")
ft_model = FastText(
    sentences=CorpusIterator(corpus_files, max_sentences=100_000),
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,
    epochs=5,
    seed=42
)
ft_model.save("ft_100.model")
print(f"fastText entrenat. Vocabulari: {len(ft_model.wv):,} paraules")

In [ ]:
# ─────────────────────────────────────────────
# fastText oficial (CommonCrawl + Wikipedia)
# ─────────────────────────────────────────────
# Descarreguem els vectors pre-entrenats en format .bin (suporta OOV)
!wget -nc https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.bin.gz
!gunzip -k cc.es.300.bin.gz

from gensim.models.fasttext import load_facebook_model
ft_official = load_facebook_model('cc.es.300.bin')
print(f"fastText oficial carregat. Vocab: {len(ft_official.wv):,}")

## 3. Càrrega dels Datasets d'Avaluació

In [ ]:
# ─────────────────────────────────────────────
# Multi-SimLex (espanyol)
# ─────────────────────────────────────────────
!wget -nc "https://web.archive.org/web/20231020014354/https://multisimlex.com/data/SPA.csv"

simlex = pd.read_csv('SPA.csv')
print(f"Multi-SimLex: {len(simlex)} parells")
print(simlex.head())

In [ ]:
# Identificar columnes rellevants
print("Columnes:", simlex.columns.tolist())

# Adaptar els noms de columna si cal (el CSV pot variar)
# Columnes típiques: 'WORD1', 'WORD2', i diverses puntuacions d'anotadors
word1_col = simlex.columns[0]   # Primera paraula
word2_col = simlex.columns[1]   # Segona paraula

# Puntuació: mitjana dels anotadors (columnes numèriques)
numeric_cols = simlex.select_dtypes(include='number').columns
simlex['score'] = simlex[numeric_cols].mean(axis=1)

print(f"\nExemple:")
print(simlex[[word1_col, word2_col, 'score']].head(10))

In [ ]:
# ─────────────────────────────────────────────
# Spanish STS
# ─────────────────────────────────────────────
from datasets import load_dataset

sts = load_dataset("PlanTL-GOB-ES/sts-es")
train_df = sts["train"].to_pandas().rename(columns={"label": "score"})
dev_df   = sts["dev"].to_pandas().rename(columns={"label": "score"})
test_df  = sts["test"].to_pandas().rename(columns={"label": "score"})

print(f"STS Train: {len(train_df)}, Dev: {len(dev_df)}, Test: {len(test_df)}")
print(train_df.head(3))

## 4. Avaluació Intrínseca: Multi-SimLex

Per a cada parell de paraules:
1. Obtenir els vectors corresponents
2. Calcular la similitud cosinus
3. Comparar amb les puntuacions humanes (correlació de Spearman)

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

def cosine_sim(v1, v2):
    """Similitud cosinus entre dos vectors."""
    v1 = v1.reshape(1, -1)
    v2 = v2.reshape(1, -1)
    return cosine_similarity(v1, v2)[0, 0]


def evaluate_intrinsic(model, simlex_df, w1_col, w2_col, model_name="model", is_fasttext=False):
    """
    Avalua un model d'embeddings sobre Multi-SimLex.
    
    Returns:
        dict amb spearman, n_oov, n_pairs
    """
    predictions = []
    gold_scores = []
    oov_pairs   = []
    
    wv = model.wv  # KeyedVectors
    
    for _, row in simlex_df.iterrows():
        w1 = str(row[w1_col]).lower()
        w2 = str(row[w2_col]).lower()
        gold = row['score']
        
        # Gestió OOV
        w1_in = w1 in wv.key_to_index
        w2_in = w2 in wv.key_to_index
        
        if w1_in and w2_in:
            sim = cosine_sim(wv[w1], wv[w2])
            predictions.append(sim)
            gold_scores.append(gold)
        elif is_fasttext:
            # fastText genera vector per a qualsevol paraula via subparaules
            vec1 = wv.get_vector(w1)
            vec2 = wv.get_vector(w2)
            sim = cosine_sim(vec1, vec2)
            predictions.append(sim)
            gold_scores.append(gold)
            if not w1_in or not w2_in:
                oov_pairs.append((w1, w2))
        else:
            # OOV sense fastText: saltem el parell
            oov_pairs.append((w1, w2))
    
    if len(predictions) < 2:
        print(f"[{model_name}] No hi ha prou parells per avaluar.")
        return None
    
    corr, pvalue = spearmanr(gold_scores, predictions)
    
    result = {
        'model':    model_name,
        'spearman': round(corr, 4),
        'p_value':  round(pvalue, 6),
        'n_pairs':  len(predictions),
        'n_oov':    len(oov_pairs),
        'oov_rate': round(len(oov_pairs) / len(simlex_df) * 100, 2)
    }
    
    print(f"[{model_name:20s}] Spearman: {corr:.4f} | Parells: {len(predictions)} | OOV: {len(oov_pairs)} ({result['oov_rate']}%)")
    return result


print("Funció d'avaluació intrínseca definida.")

In [ ]:
# Avaluar tots els models
intrinsic_results = []

# Word2Vec models
for name, model in w2v_models.items():
    r = evaluate_intrinsic(model, simlex, word1_col, word2_col, model_name=name, is_fasttext=False)
    if r: intrinsic_results.append(r)

# fastText propi
r = evaluate_intrinsic(ft_model, simlex, word1_col, word2_col, model_name='ft_100_wiki', is_fasttext=True)
if r: intrinsic_results.append(r)

# fastText oficial
r = evaluate_intrinsic(ft_official, simlex, word1_col, word2_col, model_name='ft_official_300', is_fasttext=True)
if r: intrinsic_results.append(r)

# Resum
intrinsic_df = pd.DataFrame(intrinsic_results)
print("\n=== Resum Avaluació Intrínseca (Multi-SimLex) ===")
print(intrinsic_df.sort_values('spearman', ascending=False).to_string(index=False))

In [ ]:
# Anàlisi OOV detallada per al model Word2Vec 100d
model_ref = w2v_models['w2v_100']
oov_words = []
for _, row in simlex.iterrows():
    for col in [word1_col, word2_col]:
        w = str(row[col]).lower()
        if w not in model_ref.wv.key_to_index:
            oov_words.append(w)

oov_unique = list(set(oov_words))
print(f"Paraules OOV úniques a w2v_100: {len(oov_unique)}")
print("Exemples:", oov_unique[:20])

# Comparem: el fastText oficial pot generar vectors per a totes?
ft_coverage = sum(1 for w in oov_unique if w in ft_official.wv.key_to_index)
print(f"\nD'aquestes, fastText oficial en té al vocab explícit: {ft_coverage}/{len(oov_unique)}")
print("(però fastText pot generar vectors per a TOTES via subparaules)")

## 5. Avaluació Extrínseca: Spanish STS

### 5.1 Baseline Cosinus (Mitjana Simple i TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import pearsonr

def tokenize_es(text):
    """Tokenitza simple per a espanyol."""
    text = text.lower()
    text = re.sub(r'[^a-záéíóúàèìòùüñ\s]', ' ', text)
    return text.split()


def sentence_vector_mean(tokens, wv, is_fasttext=False):
    """Vector de frase com a mitjana simple dels embeddings."""
    vectors = []
    for tok in tokens:
        if tok in wv.key_to_index:
            vectors.append(wv[tok])
        elif is_fasttext:
            vectors.append(wv.get_vector(tok))
    if not vectors:
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)


def build_tfidf_weights(df):
    """Construeix pesos TF-IDF a partir del corpus STS."""
    all_sents = list(df['sentence1']) + list(df['sentence2'])
    tfidf = TfidfVectorizer(tokenizer=tokenize_es, token_pattern=None)
    tfidf.fit(all_sents)
    return tfidf


def sentence_vector_tfidf(tokens, wv, tfidf_model, is_fasttext=False):
    """Vector de frase com a mitjana ponderada amb TF-IDF."""
    vocab = tfidf_model.vocabulary_
    idf   = tfidf_model.idf_
    vectors = []
    weights = []
    for tok in tokens:
        if tok in vocab:
            w = idf[vocab[tok]]
        else:
            w = 1.0
        if tok in wv.key_to_index:
            vectors.append(wv[tok])
            weights.append(w)
        elif is_fasttext:
            vectors.append(wv.get_vector(tok))
            weights.append(w)
    if not vectors:
        return np.zeros(wv.vector_size)
    weights = np.array(weights)
    weights = weights / weights.sum()
    return np.average(vectors, axis=0, weights=weights)


def evaluate_baseline(df, wv, model_name, aggregation='mean', tfidf_model=None, is_fasttext=False):
    """Avalua el baseline cosinus sobre un df STS."""
    preds = []
    golds = list(df['score'])
    
    for _, row in df.iterrows():
        t1 = tokenize_es(str(row['sentence1']))
        t2 = tokenize_es(str(row['sentence2']))
        
        if aggregation == 'mean':
            v1 = sentence_vector_mean(t1, wv, is_fasttext)
            v2 = sentence_vector_mean(t2, wv, is_fasttext)
        else:  # tfidf
            v1 = sentence_vector_tfidf(t1, wv, tfidf_model, is_fasttext)
            v2 = sentence_vector_tfidf(t2, wv, tfidf_model, is_fasttext)
        
        preds.append(cosine_sim(v1, v2))
    
    corr, pval = pearsonr(golds, preds)
    print(f"[{model_name:35s}] Pearson: {corr:.4f}")
    return {'model': model_name, 'pearson': round(corr, 4), 'method': 'baseline'}


print("Funcions baseline definides.")

In [ ]:
# Construïm el TF-IDF sobre el conjunt d'entrenament STS
tfidf_model = build_tfidf_weights(train_df)

sts_results = []
print("=== Baseline Cosinus (test set) ===")

# Word2Vec 100d – Mitjana simple
r = evaluate_baseline(test_df, w2v_models['w2v_100'].wv, 'w2v_100 + mean', 'mean')
sts_results.append(r)

# Word2Vec 100d – Mitjana TF-IDF
r = evaluate_baseline(test_df, w2v_models['w2v_100'].wv, 'w2v_100 + tfidf', 'tfidf', tfidf_model)
sts_results.append(r)

# fastText oficial – Mitjana simple
r = evaluate_baseline(test_df, ft_official.wv, 'ft_official + mean', 'mean', is_fasttext=True)
sts_results.append(r)

# fastText oficial – Mitjana TF-IDF
r = evaluate_baseline(test_df, ft_official.wv, 'ft_official + tfidf', 'tfidf', tfidf_model, is_fasttext=True)
sts_results.append(r)

### 5.2 Model Seqüencial Siamès (BiLSTM + Atenció)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ─────────────────────────────────────────────
# Construcció del vocabulari i matriu d'embeddings
# ─────────────────────────────────────────────

def build_vocab_and_matrix(df_list, wv, is_fasttext=False):
    """
    Construeix un vocabulari restringit a les paraules que apareixen
    al corpus STS i que existeixen al model d'embeddings.
    
    Returns:
        word2idx (dict), embedding_matrix (np.array)
    """
    # Recollir totes les paraules del corpus STS
    corpus_words = set()
    for df in df_list:
        for _, row in df.iterrows():
            for col in ['sentence1', 'sentence2']:
                corpus_words.update(tokenize_es(str(row[col])))
    
    # Tokens especials: índex 0 = PAD, índex 1 = UNK
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    dim = wv.vector_size
    
    # Matriu: primer amb zeros (PAD i UNK)
    matrix_rows = [np.zeros(dim), np.random.randn(dim) * 0.01]
    
    for word in sorted(corpus_words):
        if word in wv.key_to_index:
            word2idx[word] = len(word2idx)
            matrix_rows.append(wv[word])
        elif is_fasttext:
            word2idx[word] = len(word2idx)
            matrix_rows.append(wv.get_vector(word))
        # Si és OOV sense fastText, s'usarà <UNK>
    
    embedding_matrix = np.vstack(matrix_rows).astype(np.float32)
    print(f"Vocabulari: {len(word2idx)} paraules | Matriu: {embedding_matrix.shape}")
    return word2idx, embedding_matrix


# Usem Word2Vec 100d per al model seqüencial
word2idx, embedding_matrix = build_vocab_and_matrix(
    [train_df, dev_df, test_df],
    w2v_models['w2v_100'].wv,
    is_fasttext=False
)

In [ ]:
# ─────────────────────────────────────────────
# Dataset PyTorch per a STS
# ─────────────────────────────────────────────

class STSDataset(Dataset):
    def __init__(self, df, word2idx, max_len=64):
        self.data = df.reset_index(drop=True)
        self.word2idx = word2idx
        self.max_len = max_len
    
    def encode(self, text):
        tokens = tokenize_es(str(text))[:self.max_len]
        ids = [self.word2idx.get(t, 1) for t in tokens]  # 1 = <UNK>
        # Padding
        pad_len = self.max_len - len(ids)
        mask = [True] * len(ids) + [False] * pad_len
        ids  = ids + [0] * pad_len
        return torch.tensor(ids, dtype=torch.long), torch.tensor(mask, dtype=torch.bool)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        ids1, mask1 = self.encode(row['sentence1'])
        ids2, mask2 = self.encode(row['sentence2'])
        score = torch.tensor(float(row['score']), dtype=torch.float)
        return ids1, mask1, ids2, mask2, score


train_ds = STSDataset(train_df, word2idx)
dev_ds   = STSDataset(dev_df, word2idx)
test_ds  = STSDataset(test_df, word2idx)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
dev_dl   = DataLoader(dev_ds, batch_size=64)
test_dl  = DataLoader(test_ds, batch_size=64)

print(f"Datasets: train={len(train_ds)}, dev={len(dev_ds)}, test={len(test_ds)}")

In [ ]:
# ─────────────────────────────────────────────
# Arquitectura Siamesa BiLSTM + Atenció
# ─────────────────────────────────────────────

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x, mask):
        scores = self.proj(x).squeeze(-1)          # (B, L)
        scores = scores.masked_fill(~mask, -1e9)   # Enmascarem PAD
        alpha  = torch.softmax(scores, dim=-1)     # (B, L)
        return torch.sum(x * alpha.unsqueeze(-1), dim=1)  # (B, H)


class SiameseBiLSTMAttention(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=64, final_hidden_size=32,
                 trainable_embeddings=False):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float),
            freeze=not trainable_embeddings,
            padding_idx=0
        )
        emb_dim = embedding_matrix.shape[1]
        self.encoder = nn.LSTM(
            emb_dim, hidden_size, batch_first=True,
            bidirectional=True, dropout=0.3, num_layers=1
        )
        self.pool = AttentionPooling(hidden_size * 2)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size * 2 * 4, final_hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(final_hidden_size, 1)
        )
    
    def encode(self, input_ids, attention_mask):
        x = self.embedding(input_ids)          # (B, L, E)
        x, _ = self.encoder(x)                 # (B, L, 2H)
        return self.pool(x, attention_mask)    # (B, 2H)
    
    def forward(self, ids1, mask1, ids2, mask2):
        h1 = self.encode(ids1, mask1)
        h2 = self.encode(ids2, mask2)
        feats = torch.cat([h1, h2, torch.abs(h1 - h2), h1 * h2], dim=-1)
        return self.regressor(feats).squeeze(-1)


print("Model Siamès BiLSTM definit.")

In [ ]:
# ─────────────────────────────────────────────
# Funcions d'entrenament i avaluació
# ─────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for ids1, mask1, ids2, mask2, scores in loader:
        ids1, mask1, ids2, mask2, scores = (
            ids1.to(device), mask1.to(device),
            ids2.to(device), mask2.to(device),
            scores.to(device)
        )
        optimizer.zero_grad()
        preds = model(ids1, mask1, ids2, mask2)
        loss = criterion(preds, scores)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_golds = [], []
    for ids1, mask1, ids2, mask2, scores in loader:
        ids1, mask1, ids2, mask2 = (
            ids1.to(device), mask1.to(device),
            ids2.to(device), mask2.to(device)
        )
        preds = model(ids1, mask1, ids2, mask2).cpu().numpy()
        all_preds.extend(preds)
        all_golds.extend(scores.numpy())
    corr, _ = pearsonr(all_golds, all_preds)
    return corr


def train_siamese(model, train_dl, dev_dl, n_epochs=10, lr=1e-3, patience=3):
    """Entrena el model Siamès amb early stopping."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    
    best_dev_corr = -1
    best_state    = None
    patience_cnt  = 0
    
    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, train_dl, optimizer, criterion)
        dev_corr = evaluate_model(model, dev_dl)
        scheduler.step(-dev_corr)
        
        print(f"  Epoch {epoch:2d} | Loss: {loss:.4f} | Dev Pearson: {dev_corr:.4f}")
        
        if dev_corr > best_dev_corr:
            best_dev_corr = dev_corr
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"  Early stopping a l'epoch {epoch}.")
                break
    
    # Restaurar el millor model
    model.load_state_dict(best_state)
    return model, best_dev_corr


print("Funcions d'entrenament definides.")

In [ ]:
# Experiment 1: Embeddings congelats (freeze=True)
print("\n=== Model BiLSTM – Embeddings Congelats ===")
model_frozen = SiameseBiLSTMAttention(
    embedding_matrix,
    hidden_size=64,
    final_hidden_size=32,
    trainable_embeddings=False
)
model_frozen, best_dev = train_siamese(model_frozen, train_dl, dev_dl, n_epochs=15)
test_corr_frozen = evaluate_model(model_frozen, test_dl)
print(f"\nTest Pearson (congelat): {test_corr_frozen:.4f}")
sts_results.append({'model': 'BiLSTM_frozen', 'pearson': round(test_corr_frozen, 4), 'method': 'sequential'})

In [ ]:
# Experiment 2: Embeddings entrenables (freeze=False)
print("\n=== Model BiLSTM – Embeddings Entrenables ===")
model_trainable = SiameseBiLSTMAttention(
    embedding_matrix,
    hidden_size=64,
    final_hidden_size=32,
    trainable_embeddings=True
)
model_trainable, best_dev = train_siamese(model_trainable, train_dl, dev_dl, n_epochs=15)
test_corr_trainable = evaluate_model(model_trainable, test_dl)
print(f"\nTest Pearson (entrenable): {test_corr_trainable:.4f}")
sts_results.append({'model': 'BiLSTM_trainable', 'pearson': round(test_corr_trainable, 4), 'method': 'sequential'})

In [ ]:
# Experiment 3: Embeddings aleatoris (sense pre-entrenament)
print("\n=== Model BiLSTM – Embeddings Aleatoris ===")

# Matriu aleatòria (mateixa dimensió)
random_matrix = np.random.randn(*embedding_matrix.shape).astype(np.float32) * 0.01
random_matrix[0] = 0.0  # PAD = zeros

model_random = SiameseBiLSTMAttention(
    random_matrix,
    hidden_size=64,
    final_hidden_size=32,
    trainable_embeddings=True  # Han de ser entrenables
)
model_random, best_dev = train_siamese(model_random, train_dl, dev_dl, n_epochs=15)
test_corr_random = evaluate_model(model_random, test_dl)
print(f"\nTest Pearson (aleatori): {test_corr_random:.4f}")
sts_results.append({'model': 'BiLSTM_random_emb', 'pearson': round(test_corr_random, 4), 'method': 'sequential'})

### 5.3 Model BERT Siamès (BETO)

In [ ]:
from transformers import AutoTokenizer, AutoModel

BERT_MODEL = "dccuchile/bert-base-spanish-wwm-cased"  # BETO
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)


class STSBertDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.data = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def encode(self, text):
        enc = self.tokenizer(
            str(text),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        ids1, mask1 = self.encode(row['sentence1'])
        ids2, mask2 = self.encode(row['sentence2'])
        score = torch.tensor(float(row['score']), dtype=torch.float)
        return ids1, mask1, ids2, mask2, score


bert_train_ds = STSBertDataset(train_df, bert_tokenizer)
bert_dev_ds   = STSBertDataset(dev_df, bert_tokenizer)
bert_test_ds  = STSBertDataset(test_df, bert_tokenizer)

bert_train_dl = DataLoader(bert_train_ds, batch_size=16, shuffle=True)
bert_dev_dl   = DataLoader(bert_dev_ds, batch_size=32)
bert_test_dl  = DataLoader(bert_test_ds, batch_size=32)

print("Datasets BERT creats.")

In [ ]:
class MeanPooling(nn.Module):
    def forward(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        x = last_hidden_state * mask
        return x.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


class BETOSiameseRegressor(nn.Module):
    def __init__(self, model_name=BERT_MODEL, final_hidden_size=32, freeze_bert=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        
        # Congelar BERT si es vol (útil per a debugging ràpid)
        if freeze_bert:
            for param in self.encoder.parameters():
                param.requires_grad = False
        
        self.pool = MeanPooling()
        hidden = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden * 4, final_hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(final_hidden_size, 1)
        )
    
    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.pool(out.last_hidden_state, attention_mask)
    
    def forward(self, ids1, mask1, ids2, mask2):
        h1 = self.encode(ids1, mask1)
        h2 = self.encode(ids2, mask2)
        feats = torch.cat([h1, h2, torch.abs(h1 - h2), h1 * h2], dim=-1)
        return self.regressor(feats).squeeze(-1)


print("Model BETO Siamès definit.")

In [ ]:
print("\n=== Model BETO Siamès ===")
bert_model = BETOSiameseRegressor(final_hidden_size=32, freeze_bert=False)

# Entrenament BERT: lr molt baix per a fine-tuning
bert_model = bert_model.to(device)
optimizer_bert = torch.optim.AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.MSELoss()

best_dev_corr = -1
best_bert_state = None

for epoch in range(1, 6):  # Fine-tuning: pocs epochs
    bert_model.train()
    total_loss = 0
    for ids1, mask1, ids2, mask2, scores in tqdm(bert_train_dl, desc=f"Epoch {epoch}"):
        ids1, mask1, ids2, mask2, scores = (
            ids1.to(device), mask1.to(device),
            ids2.to(device), mask2.to(device),
            scores.to(device)
        )
        optimizer_bert.zero_grad()
        preds = bert_model(ids1, mask1, ids2, mask2)
        loss = criterion(preds, scores)
        loss.backward()
        nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer_bert.step()
        total_loss += loss.item()
    
    dev_corr = evaluate_model(bert_model, bert_dev_dl)
    print(f"  Epoch {epoch} | Loss: {total_loss/len(bert_train_dl):.4f} | Dev Pearson: {dev_corr:.4f}")
    
    if dev_corr > best_dev_corr:
        best_dev_corr = dev_corr
        best_bert_state = {k: v.cpu().clone() for k, v in bert_model.state_dict().items()}

bert_model.load_state_dict(best_bert_state)
test_corr_bert = evaluate_model(bert_model, bert_test_dl)
print(f"\nTest Pearson (BETO): {test_corr_bert:.4f}")
sts_results.append({'model': 'BETO_siamese', 'pearson': round(test_corr_bert, 4), 'method': 'bert'})

## 6. Anàlisi de Resultats

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────
# Taula resum STS extrínseca
# ─────────────────────────────────────────────
sts_df = pd.DataFrame(sts_results).sort_values('pearson', ascending=False)
print("=== Resultats STS (Pearson, test set) ===")
print(sts_df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# Taula resum intrínseca
# ─────────────────────────────────────────────
print("\n=== Resultats Multi-SimLex (Spearman) ===")
print(intrinsic_df.sort_values('spearman', ascending=False).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# Gràfics de comparació
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Comparació de Models d\'Embeddings', fontsize=14, fontweight='bold')

# Esquerra: Multi-SimLex (Spearman)
ax = axes[0]
colors_intr = ['#2196F3', '#2196F3', '#2196F3', '#FF9800', '#4CAF50']
df_sorted_intr = intrinsic_df.sort_values('spearman')
bars = ax.barh(df_sorted_intr['model'], df_sorted_intr['spearman'], color=colors_intr[:len(df_sorted_intr)])
ax.set_xlabel('Spearman r')
ax.set_title('Avaluació Intrínseca\n(Multi-SimLex)')
ax.set_xlim(0, 1)
for bar, val in zip(bars, df_sorted_intr['spearman']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

# Dreta: STS (Pearson)
ax = axes[1]
method_colors = {'baseline': '#2196F3', 'sequential': '#FF9800', 'bert': '#4CAF50'}
df_sorted_sts = sts_df.sort_values('pearson')
bar_colors = [method_colors[m] for m in df_sorted_sts['method']]
bars = ax.barh(df_sorted_sts['model'], df_sorted_sts['pearson'], color=bar_colors)
ax.set_xlabel('Pearson r')
ax.set_title('Avaluació Extrínseca\n(Spanish STS)')
ax.set_xlim(0, 1)
for bar, val in zip(bars, df_sorted_sts['pearson']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

# Llegenda
patches = [mpatches.Patch(color=c, label=l) for l, c in method_colors.items()]
axes[1].legend(handles=patches, loc='lower right', title='Mètode')

plt.tight_layout()
plt.savefig('results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gràfic guardat a 'results_comparison.png'")

In [ ]:
# ─────────────────────────────────────────────
# Anàlisi: Efecte de la dimensionalitat (Word2Vec)
# ─────────────────────────────────────────────

dims = [25, 50, 100]
spearman_by_dim = [intrinsic_df[intrinsic_df['model'] == f'w2v_{d}']['spearman'].values[0]
                   for d in dims if f'w2v_{d}' in intrinsic_df['model'].values]

plt.figure(figsize=(6, 4))
plt.plot(dims[:len(spearman_by_dim)], spearman_by_dim, 'o-', color='#2196F3', linewidth=2, markersize=8)
plt.xlabel('Dimensió dels Embeddings')
plt.ylabel('Spearman r (Multi-SimLex)')
plt.title('Efecte de la Dimensionalitat (Word2Vec)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dim_effect.png', dpi=150)
plt.show()

## 7. Anàlisi Qualitativa i Discussió

### 7.1 Efecte de la Dimensionalitat

S'observa que... *(completar amb els valors reals obtinguts)*

### 7.2 Word2Vec vs. fastText (OOV)

fastText pot generar vectors per a paraules fora de vocabulari gràcies als subword n-grams. Això és especialment útil per a:
- Paraules morfològicament riques (espanyol té moltes flexions)
- Paraules rares o especialitzades

### 7.3 Guany del model Seqüencial

El pas de la mitjana de vectors (baseline) al model BiLSTM permet capturar ordre i context dins de la frase.

### 7.4 Guany de BERT

BERT proporciona representacions contextuals: la mateixa paraula té un vector diferent en contextos diferents.

In [ ]:
# ─────────────────────────────────────────────
# OPCIONAL: Anàlisi d'OOV i cobertura lèxica
# ─────────────────────────────────────────────

def analyze_oov(df_list, wv, model_name):
    total_tokens = 0
    oov_tokens   = 0
    oov_words    = set()
    
    for df in df_list:
        for _, row in df.iterrows():
            for col in ['sentence1', 'sentence2']:
                tokens = tokenize_es(str(row[col]))
                for tok in tokens:
                    total_tokens += 1
                    if tok not in wv.key_to_index:
                        oov_tokens += 1
                        oov_words.add(tok)
    
    print(f"\n[{model_name}]")
    print(f"  Tokens totals: {total_tokens:,}")
    print(f"  Tokens OOV: {oov_tokens:,} ({oov_tokens/total_tokens*100:.1f}%)")
    print(f"  Types OOV únics: {len(oov_words):,}")
    print(f"  Exemples OOV: {list(oov_words)[:10]}")


dfs = [train_df, dev_df, test_df]
analyze_oov(dfs, w2v_models['w2v_100'].wv, 'Word2Vec 100d')
analyze_oov(dfs, ft_model.wv,              'fastText 100d wiki')
analyze_oov(dfs, ft_official.wv,           'fastText oficial 300d')

In [ ]:
# ─────────────────────────────────────────────
# OPCIONAL: Robustesa davant errors tipogràfics
# ─────────────────────────────────────────────

import random

def perturb_text(text, error_rate=0.15):
    """Genera una versió pertorbada del text amb errors tipogràfics."""
    tokens = text.split()
    perturbed = []
    for token in tokens:
        if random.random() < error_rate and len(token) > 3:
            # Tipus d'error aleatori
            error_type = random.choice(['swap', 'delete', 'accent'])
            
            if error_type == 'swap' and len(token) > 2:
                # Intercanviar dos caràcters adjacents
                i = random.randint(0, len(token) - 2)
                token = token[:i] + token[i+1] + token[i] + token[i+2:]
            
            elif error_type == 'delete':
                # Eliminar un caràcter
                i = random.randint(1, len(token) - 1)
                token = token[:i] + token[i+1:]
            
            elif error_type == 'accent':
                # Eliminar accents (error típic en espanyol)
                accent_map = {'á':'a', 'é':'e', 'í':'i', 'ó':'o', 'ú':'u', 'ñ':'n'}
                token = ''.join(accent_map.get(c, c) for c in token)
        
        perturbed.append(token)
    return ' '.join(perturbed)


# Crear conjunt de test pertorbat
random.seed(42)
test_perturbed = test_df.copy()
test_perturbed['sentence1'] = test_perturbed['sentence1'].apply(lambda t: perturb_text(str(t)))
test_perturbed['sentence2'] = test_perturbed['sentence2'].apply(lambda t: perturb_text(str(t)))

# Exemple de pertorbació
print("ORIGINAL:", test_df['sentence1'].iloc[0])
print("PERTORBAT:", test_perturbed['sentence1'].iloc[0])

# Avaluar robustesa: Word2Vec vs fastText oficial
print("\n=== Robustesa (Test Pertorbat) ===")
evaluate_baseline(test_perturbed, w2v_models['w2v_100'].wv, 'w2v_100 + mean [PERTURB]', 'mean')
evaluate_baseline(test_perturbed, ft_official.wv, 'ft_official + mean [PERTURB]', 'mean', is_fasttext=True)

In [ ]:
# ─────────────────────────────────────────────
# Resum final
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("RESUM FINAL – Pràctica 4")
print("="*60)

print("\n[Avaluació Intrínseca – Multi-SimLex] Spearman r")
print(intrinsic_df[['model','spearman','oov_rate']].sort_values('spearman', ascending=False).to_string(index=False))

print("\n[Avaluació Extrínseca – Spanish STS] Pearson r")
print(sts_df[['model','method','pearson']].sort_values('pearson', ascending=False).to_string(index=False))